In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(
    ".venv/01-01_철강_공정_개관_설비태그.csv"
)
# df.info()

In [ ]:
# tag = "PL1-SNT-FAN-01-ViB"
# 공장-공정-설비-일련번호-계측값

# parts = tag.split('-')
# print(parts)

# 하이푼 - 기준으로 나눈 결과 문자열을 변수에 따로 저장
# plant = parts[0] # 공장
# process = parts[1] # 공정
# equip = parts[2] # 설비
# unit_no = parts[3] # 일련번호
# measure = parts[4] # 계측값

# print(plant, process, equip, unit_no, measure)
# PL1 SNT FAN 01 ViB

# 공정 데이터 규칙
# PROCESS_KR = {
#     "SNT" : "소결",
#     "CKO" : "코크스",
#     "BF" : "고로",
#     "BOF" : "전로",
#     "CCM" : "연주",
#     "HSM" : "열간압연",
#     "CRM" : "냉간압연",
#     "UTL" : "유틸리티",
# }

# print(df["tag"].value_counts())

# print(PROCESS_KR["BOF"])
# print(PROCESS_KR.get('BOF', '미등록')) # 없는 태그를 가져오는 것 방지



# print(MEASURE_KR.get('VIB', '미등록'))

['PL1', 'SNT', 'FAN', '01', 'ViB']
PL1 SNT FAN 01 ViB
진동


In [ ]:
# print(df.shape) # (24, 4)
# print(df.columns) # Index(['tag', 'unit', 'sample_value', 'note'], dtype='str')
# print(df.columns.tolist()) # ['tag', 'unit', 'sample_value', 'note']


(24, 4)
Index(['tag', 'unit', 'sample_value', 'note'], dtype='str')
['tag', 'unit', 'sample_value', 'note']


In [ ]:
# 공정별로 몇 개의 태그가 있는지 세어보기
# 고로 N개
# 냉간압연 Nro




In [ ]:
PROCESS_KR = {
    "SNT": "소결",
    "CKO": "코크스",
    "BF": "고로",
    "BOF": "전로",
    "CCM": "연주",
    "HSM": "열간압연",
    "CRM": "냉간압연",
    "UTL": "유틸리티",
}

PROCESS_KR2 = {
    "SNT": "상공정",
    "CKO": "상공정",
    "BF": "상공정",
    "BOF": "상공정",
    "CCM": "상공정",
    "HSM": "하공정",
    "CRM": "하공정",
    "UTL": "유틸리티",
}

MEASURE_KR = {
    "VIB": "진동",
    "CUR": "전류",
    "TMP": "온도",
    "PRS": "압력",
    "FLW": "유량",
    "SPD": "속도",
    "LVL": "레벨"
}


# 태그에서 공정코드와 계측항목 추출
df["process"] = df["tag"].str.split("-").str[1]
df["measure"] = df["tag"].str.split("-").str[-1]


# 한글명으로 변환
df["process_kr"] = df["process"].map(PROCESS_KR)
df["process_kr2"] = df["process"].map(PROCESS_KR2)
df["measure_kr"] = df["measure"].map(MEASURE_KR)


# 1. 상공정, 하공정, 유틸리티 개수
print("1. 공정구분별 개수")
print(df["process_kr2"].value_counts())


# 2. 공정별 태그 개수
print("\n2. 공정별 태그 개수")
print(df["process_kr"].value_counts())

# 가장 많이 등장하는 공정
process_counts = df["process_kr"].value_counts()

print("가장 많이 등장하는 공정")
print(process_counts[process_counts == process_counts.max()])


# 3. 계측항목별 태그 개수
print("\n3. 계측항목별 태그 개수")
print(df["measure_kr"].value_counts())

# 가장 많이 등장하는 계측항목
measure_counts = df["measure_kr"].value_counts()

print("가장 많이 등장하는 계측항목")
print(measure_counts[measure_counts == measure_counts.max()])

process_kr2
상공정     14
하공정      7
유틸리티     3
Name: count, dtype: int64
process_kr
고로      4
열간압연    4
소결      3
연주      3
냉간압연    3
유틸리티    3
코크스     2
전로      2
Name: count, dtype: int64
고로
measure_kr
온도    6
전류    5
압력    5
진동    4
유량    3
속도    1
Name: count, dtype: int64
온도


In [ ]:
import pandas as pd

df = pd.read_csv(".venv/01-02_원료_전처리와_제선_제선조업.csv")

# print(df.head())
# print(df.isna().sum().sum())

# 1. csv에서 datetime 데이터 불러오기(to_datetime 이용)
print(df["timestamp"].dtype)  # str
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(df["timestamp"].dtype)  # datetime64

# 2. read_csv()의 옵션값 이용(parse_dates=["timestamp")
df = pd.read_csv(
    ".venv/01-02_원료_전처리와_제선_제선조업.csv", parse_dates=["timestamp"]
)
print(df["timestamp"].dtype)  # datetime64

# timestamp의 시간 간격을 알아보기
gaps = df["timestamp"].diff().value_counts()
print(gaps)
# timestamp
# 0 days 00:01:00    719
# Name: count, dtype: int64

# 송풍량, 송풍압, 송풍기 진동
print(
    df[["blast_flow_nm3min", "blast_pressure_kpa", "blower_vib_mms"]]
    .describe()
    .round(1)
)

# 이동평균 : N분 간의 흔들림을 확인하여 송풍량의 장기적인 방향을 보는 지표
# 통기성이 나빠지면 공기가 원료층을 통과하기 어려워져서 실제 들어가는 풍량이 감소할 수 있습니다.

df["flow_ma"] = df["blast_flow_nm3min"].rolling(window=15).mean()  # 15분 간격 이동평균
# print(df["flow_ma"].head(17))

print(
    df["flow_ma"].iloc[14].round(1), df["flow_ma"].iloc[400].round(1)
)  # 5201.5 5200.8 -> 이 값들은 차이가 크지 않음. 이 값으로는 통기성 악화가 보이지 않음
# 통풍량으로는 현재 csv에서 통기성 악화를 확인할 수 없음

# 이동 표준편차
df["top_sd"] = df["top_pressure_kpa"].rolling(window=30).std()  # 30분 간격 이동표준편차
print(
    df["top_sd"].iloc[200].round(1), df["top_sd"].iloc[400].round(1)
)  # 5201.5 5200.8 -> 이 값들은 차이가 크지 않음. 이 값으로는 통기성 악화가 보이지 않음
